In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

df = pd.read_csv("data/raw/churn.csv")
print(f"Original shape: {df.shape}")

Original shape: (10000, 14)


In [2]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
df.describe()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [6]:
df.dtypes


RowNumber            int64
CustomerId           int64
Surname             object
CreditScore          int64
Geography           object
Gender              object
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object

In [7]:
df.isnull().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [8]:
df.duplicated().sum()

0

In [10]:
df['Geography'].unique()

array(['France', 'Spain', 'Germany'], dtype=object)

In [11]:
df['Exited'].unique()

array([1, 0], dtype=int64)

In [3]:
# 1. INJECT MISSING VALUES
age_null_idx = df.sample(frac=0.05, random_state=1).index
df.loc[age_null_idx, "Age"] = np.nan

balance_null_idx = df.sample(frac=0.08, random_state=2).index
df.loc[balance_null_idx, "Balance"] = np.nan

credit_null_idx = df.sample(frac=0.03, random_state=3).index
df.loc[credit_null_idx, "CreditScore"] = np.nan

print(f"Nulls injected:")
print(f"  Age         : {df['Age'].isna().sum()}")
print(f"  Balance     : {df['Balance'].isna().sum()}")
print(f"  CreditScore : {df['CreditScore'].isna().sum()}")

Nulls injected:
  Age         : 500
  Balance     : 800
  CreditScore : 300


In [11]:
df['Balance'].isnull().sum()

800

In [8]:
df.isnull().sum()

RowNumber            0
CustomerId           0
Surname              0
CreditScore        300
Geography            0
Gender               0
Age                500
Tenure               0
Balance            800
NumOfProducts        0
HasCrCard            0
IsActiveMember       0
EstimatedSalary      0
Exited               0
dtype: int64

In [4]:
# 2. INJECT STRING INCONSISTENCIES
geo_mess_idx = df.sample(frac=0.40, random_state=4).index
def mess_geography(val):
    options = {
        "France" : ["france", "FRANCE", "France "],
        "Germany": ["germany", "GERMANY", "Germany "],
        "Spain"  : ["spain", "SPAIN", "Spain "]
    }
    return np.random.choice(options.get(val, [val]))
df.loc[geo_mess_idx, "Geography"] = df.loc[geo_mess_idx, "Geography"].apply(mess_geography)

gender_mess_idx = df.sample(frac=0.40, random_state=5).index
def mess_gender(val):
    options = {
        "Male"  : ["male", "MALE", "M"],
        "Female": ["female", "FEMALE", "F"]
    }
    return np.random.choice(options.get(val, [val]))
df.loc[gender_mess_idx, "Gender"] = df.loc[gender_mess_idx, "Gender"].apply(mess_gender)

print(f"\nString issues injected:")
print(f"  Geography unique: {df['Geography'].unique()}")
print(f"  Gender unique   : {df['Gender'].unique()}")


String issues injected:
  Geography unique: ['France' 'SPAIN' 'France ' 'france' 'spain' 'Spain' 'Germany' 'FRANCE'
 'GERMANY' 'Spain ' 'Germany ' 'germany']
  Gender unique   : ['Female' 'F' 'female' 'Male' 'MALE' 'male' 'FEMALE' 'M']


In [15]:
# 3. INJECT OUTLIERS
age_out_idx = df.dropna(subset=["Age"]).sample(n=25, random_state=6).index
df.loc[age_out_idx, "Age"] = np.random.choice([0, 1, 120, 135, 150], size=25)

credit_out_idx = df.dropna(subset=["CreditScore"]).sample(n=25, random_state=7).index
df.loc[credit_out_idx, "CreditScore"] = np.random.choice([10, 50, 900, 950, 999], size=25)

salary_out_idx = df.sample(n=10, random_state=8).index
df.loc[salary_out_idx, "EstimatedSalary"] = np.random.uniform(-50000, -1000, size=10)

print(f"\nOutliers injected:")
print(f"  Age impossible        : {df[(df['Age'] < 18) | (df['Age'] > 100)].shape[0]} rows")
print(f"  CreditScore out-range : {df[(df['CreditScore'] < 300) | (df['CreditScore'] > 850)].shape[0]} rows")
print(f"  Negative salary       : {df[df['EstimatedSalary'] < 0].shape[0]} rows")


Outliers injected:
  Age impossible        : 25 rows
  CreditScore out-range : 25 rows
  Negative salary       : 10 rows


In [5]:
# 4. INJECT DUPLICATES
dup_rows = df.sample(n=150, random_state=9)
df = pd.concat([df, dup_rows], ignore_index=True)
print(f"\nDuplicates injected:")
print(f"  Shape after duplicates: {df.shape}")

# 5. INJECT DTYPE ISSUES
df["HasCrCard"]      = df["HasCrCard"].astype(float)
df["IsActiveMember"] = df["IsActiveMember"].astype(float)
print(f"\nDtype issues injected:")
print(f"  HasCrCard dtype      : {df['HasCrCard'].dtype}")
print(f"  IsActiveMember dtype : {df['IsActiveMember'].dtype}")


Duplicates injected:
  Shape after duplicates: (10150, 14)

Dtype issues injected:
  HasCrCard dtype      : float64
  IsActiveMember dtype : float64


In [17]:
# SAVE DIRTY DATASET
df.to_csv("data/raw/churn_dirty.csv", index=False)
print(f"\n{'='*45}")
print(f"churn_dirty.csv saved to data/raw/")
print(f"Final shape : {df.shape}")
print(f"Total nulls : {df.isnull().sum().sum()}")
print(f"{'='*45}")


churn_dirty.csv saved to data/raw/
Final shape : (10150, 14)
Total nulls : 1614


In [7]:
df.isnull().sum()

RowNumber            0
CustomerId           0
Surname              0
CreditScore        304
Geography            0
Gender               0
Age                504
Tenure               0
Balance            806
NumOfProducts        0
HasCrCard            0
IsActiveMember       0
EstimatedSalary      0
Exited               0
dtype: int64